<div style="display:flex; align-items:center; gap:10px; margin-bottom:8px;">
  <span style="font-size:26px; color:#9558B2;">●</span>
  <span style="font-size:26px; color:#389826;">●</span>
  <span style="font-size:26px; color:#CB3C33;">●</span>
  <span style="font-size:26px; color:#4063D8;">●</span>
  <span style="font-size:30px; font-weight:700; margin-left:6px;">Julia</span>
</div>

# Julia с нуля — **Lesson 9**
## 📘 **Julia is Fast** — JIT-компиляция, специализация, типовая стабильность и измерение производительности

**Cartesian School · Julia Course**  
**Автор:** Siergej Sobolewski  
**Copyright:** © 2026 Cartesian School


## Информация об уроке

| Поле | Значение |
|---|---|
| Курс | Julia с нуля |
| Номер урока | Lesson 9 |
| Название | Julia is Fast |
| Уровень | Начальный+ / średniozaawansowany |
| Ориентировочное время | 240–300 minut |
| Требования | Lesson 0–8 |
| Темы | JIT, `BenchmarkTools`, `@time`, `@btime`, аллокации, стабильность типов, `@code_warnтипe`, C przez `ccall`, Python i NumPy, SIMD, `@inbounds`, cache locality, column-major order, interpretacja результатów |
| Автор | Siergej Sobolewski |
| Права | © 2026 Cartesian School |


## Plan lekcji

1. Co znaczy „Julia jest szybka”.
2. Funkcja testowa `sum`.
3. Przygotowanie danych.
4. Dlaczego benchmark wymaga metodologii.
5. JIT i pierwszy przebieg.
6. `@time`.
7. `BenchmarkTools`.
8. `@btime`, `@belapsed`, `@benchmark`.
9. Interpolacja `$`.
10. Globalne zmienne.
11. Alokacje.
12. Stabilność типów.
13. `@code_warnтипe`.
14. Julia `sum`.
15. Ręczna implementacja Julia.
16. SIMD.
17. `@inbounds`.
18. Arytmetyka zmiennoprzecinkowa.
19. C przez `ccall`.
20. C `-O3`.
21. `-ffast-math`.
22. Python built-in `sum`.
23. NumPy.
24. Ręczna цикл Python.
25. Uczciwe porównanie.
26. Column-major order.
27. Broadcasting i fuzja.
28. Views i аллокации.
29. Benchmark funkcji mutujących.
30. Statystyka benchmarków.
31. Pułapki микробенчмаркów.
32. Практика.
33. Мини-проект.
34. Checkpoint.
35. Итоги.


## Учебный стандарт Cartesian School

| Oznaczenie | Znaczenie |
|---|---|
| **Цель** | czego nauczysz się w danym fragmencie |
| **Теория** | definicje i reguły |
| **Пример** | minimalny, działający код |
| **Анализ** | wyjaśnienie działania |
| **Важно** | zasada wymagająca uwagi |
| **Типичная ошибка** | częsty ошибка i jego przyczyna |
| **Попробуйте сами** | mały eksperyment |
| **Практика** | zadanie do samodzielnego wykonania |
| **Итоги** | najważniejsze wnioski |


## Цели урока

После завершения Lesson 9 вы сможете:

- oddzielić koszt kompilacji JIT od времяu wykonania;
- korzystać z `BenchmarkTools`;
- interpretować результатi benchmarków;
- измерять аллокации;
- rozpoznawać problemy ze stabilnością типów;
- porównywać implementacje Julia, C, Python i NumPy;
- wyjaśnić rolę SIMD, `@inbounds` i lokalności pamięci;
- unikać najczęstszych błędów metodologicznych w benchmarkach.


## **1. Co naprawdę znaczy „Julia jest szybka”?**

### Теория

Julia została zaprojektowana tak, aby код wysokiego poziomu mógł być kompilowany do bardzo wydajnego кодu maszynowego.

Nie oznacza to jednak, że każdy program Julia automatycznie będzie быстрый.

Wydajność zależy m.in. od:

- algorytmu,
- типów danych,
- stabilności типów,
- числа alokacji,
- dostępu do pamięci,
- możliwości SIMD,
- jakości benchmarku.


### Важно

Benchmark nie powinien „udowadniać”, że konkretny język jest najszybszy.

Powinien odpowiadać na pytanie:

> Jak szybko wykonuje się **konkretna implementacja konkretnego zadania** w określonym środowisku?


## **2. Funkcja testowa — suma elementów**

Rozpatrujemy:

\[
\mathrm{sum}(a)=\sum_{i=1}^{n}a_i
\]

To dobry przykład, ponieważ algorytm jest prosty i łatwo zaimplementować go w kilku językach.


### Пример — przygotowanie danych


In [ ]:
using Random

Random.seed!(42)

const N = 10^7
a = rand(Float64, N)

@show length(a)
@show eltype(a)


Dla rozkładu jednostajnego `U(0,1)` значение oczekiwana pojedynczego elementu wynosi `0.5`, więc suma dla `10^7` elementów powinna być bliska `5 × 10^6`.


In [ ]:
julia_sum = sum(a)

@show julia_sum
@show julia_sum / N


## **3. Dlaczego pojedynczy pomiar jest niewystarczający?**

Время выполнения zależy m.in. od:

- kompilacji JIT,
- cache CPU,
- planisty systemu,
- częstotliwości CPU,
- procesów w tle,
- garbage collectora.

Dlatego profesjonalny benchmark wymaga wielu próbek.


## **4. JIT i pierwszy przebieg**

Pierwsze вызов funkcji może obejmować kompilację specjalizacji dla konkretnych типów аргументów.


In [ ]:
function sum_loop(values)
    total = zero(eltype(values))
    for x in values
        total += x
    end
    return total
end

small = rand(1000)

@time sum_loop(small)
@time sum_loop(small)


### Анализ

Pierwszy pomiar może zawierać koszt kompilacji. Kolejny jest bliższy kosztowi samego wykonania.


## **5. Makro `@time`**

`@time` jest wygodne do быстрыйej diagnostyki:


In [ ]:
@time sum(a)


### Важно

`@time` jest użyteczne, ale do микробенчмаркów preferujemy `BenchmarkTools`.


## **6. `BenchmarkTools.jl`**

Jeżeli pakiet nie jest jeszcze w środowisku:

```text
pkg> add BenchmarkTools
```

W notebooku kursowym nie instalujemy go automatycznie.


In [ ]:
# Odkomentuj, jeżeli BenchmarkTools jest dostępny:
# using BenchmarkTools


## **7. `@btime`, `@belapsed`, `@benchmark`**

Najczęściej używane narzędzia:

| Narzędzie | Zastosowanie |
|---|---|
| `@btime` | krótki результат времяu i alokacji |
| `@belapsed` | время jako число sekund |
| `@benchmark` | pełny rozkład результатów |


In [ ]:
# using BenchmarkTools
# @btime sum($a)


In [ ]:
# elapsed = @belapsed sum($a)
# @show elapsed


In [ ]:
# trial = @benchmark sum($a)
# trial


## **8. Interpolacja `$` w benchmarkach**

W `BenchmarkTools` dane z zewnętrznego zakresu zwykle interpolujemy:

```julia
@btime sum($a)
```

zamiast:

```julia
@btime sum(a)
```

Dzięki temu benchmark nie mierzy przypadkowego kosztu dostępu do zmiennej globalnej.


## **9. Globalne zmienne a производительность**

Kod krytyczny производительностиowo powinien zwykle znajdować się wewnątrz funkcji.


In [ ]:
function manual_sum(values)
    total = zero(eltype(values))
    for x in values
        total += x
    end
    return total
end

@assert manual_sum(a) ≈ sum(a)


## **10. Alokacje pamięci**

Julia pozwala zизмерять ilość zaalokowanej pamięci:


In [ ]:
@show @allocated manual_sum(a)


### Важно

Nie każda аллокация jest zła. Problemem są głównie zbędne аллокации w gorących циклch oraz duże obiekty tymвремяowe.


## **11. Stabilność типów**

Funkcja jest типe-stable, jeżeli компилятор może przewidzieć тип результатu na podstawie типów аргументów.


### Пример stabilny


In [ ]:
function stable_example(x)
    if x > 0
        return x
    else
        return -x
    end
end


### Пример niestabilny


In [ ]:
function unstable_example(x)
    if x > 0
        return x
    else
        return "negative"
    end
end


W drugiej funkcji результат może być liczbą albo napisem, co utrudnia optymalizację кодu korzystającego z результатu.


## **12. `@code_warnтипe`**

To jedno z podstawowych narzędzi diagnostycznych:


In [ ]:
@code_warntype stable_example(5)


In [ ]:
@code_warntype unstable_example(5)


## **13. Wbudowane `sum` w Julia**

Wbudowana implementacja jest dobrym punktem odniesienia.


In [ ]:
# using BenchmarkTools
# bench_julia_builtin = @benchmark sum($a)
# minimum(bench_julia_builtin)


## **14. Ręczna implementacja Julia**


In [ ]:
function mysum(values)
    total = zero(eltype(values))
    for x in values
        total += x
    end
    return total
end

@assert mysum(a) ≈ sum(a)


In [ ]:
# using BenchmarkTools
# @btime mysum($a)


### Анализ

Ręczna цикл w Julia nie jest z definicji wolna. Kompilator może przekształcić ją do wydajnego кодu maszynowego, szczególnie gdy типy są stabilne.


## **15. SIMD**

SIMD oznacza wykonywanie jednej instrukcji na wielu elementach danych.

Julia udostępnia makro `@simd`, które może pomóc компиляторowi w векторyzacji odpowiedniej pętli.


In [ ]:
function mysum_simd(values)
    total = zero(eltype(values))
    @simd for i in eachindex(values)
        total += values[i]
    end
    return total
end

@assert mysum_simd(a) ≈ sum(a)


In [ ]:
# using BenchmarkTools
# @btime mysum_simd($a)


### Важно

`@simd` nie jest magicznym przyspieszaczem. Należy stosować go tylko wtedy, gdy semantyka pętli pozwala na bezpieczną векторyzację.


## **16. `@inbounds`**

Normalnie Julia sprawdza poprawność indeksów tablic. W gorących циклch można świadomie wyłączyć te sprawdzenia.


In [ ]:
function mysum_inbounds(values)
    total = zero(eltype(values))
    @inbounds for i in eachindex(values)
        total += values[i]
    end
    return total
end

@assert mysum_inbounds(a) ≈ sum(a)


### Важно

`@inbounds` usuwa mechanizm bezpieczeństwa. Używaj go dopiero wtedy, gdy poprawność indeksów jest pewna.


## **17. Połączenie `@inbounds` i `@simd`**


In [ ]:
function mysum_fast(values)
    total = zero(eltype(values))
    @inbounds @simd for i in eachindex(values)
        total += values[i]
    end
    return total
end

@assert mysum_fast(a) ≈ sum(a)


## **18. Arytmetyka zmiennoprzecinkowa i kolejność sumowania**

Dodawanie `Float64` nie jest idealnie łączne matematycznie:

```text
(a + b) + c
```

może różnić się minimalnie od:

```text
a + (b + c)
```

Dlatego dwie быстрыйe implementacje mogą zwrócić результатi różniące się na poziomie błędu zaokrągleń.


In [ ]:
x = [1e16, 1.0, -1e16]

@show (x[1] + x[2]) + x[3]
@show x[1] + (x[2] + x[3])


### Хорошая практика

Przy porównywaniu результатów zmiennoprzecinkowych używaj `≈` / `isapprox`, jeżeli oczekiwane są niewielkie różnice numeryczne.


## **19. C przez `ccall`**

Oryginalny notebook porównywał Julia z ręczną implementacją C kompilowaną do biblioteki dynamicznej.

Poniższy przykład jest opcjonalny i wymaga działającego компиляторa C.


In [ ]:
using Libdl

c_source = raw"""
#include <stddef.h>

double c_sum(size_t n, const double *x) {
    double s = 0.0;
    for (size_t i = 0; i < n; ++i) {
        s += x[i];
    }
    return s;
}
"""

# Kod kompilacji jest celowo zakomentowany.
# Wymaga systemowego kompilatora, np. gcc/clang.


### Dlaczego przykład jest opcjonalny?

Notebook powinien działać również w środowisku bez systemowego компиляторa C.

Benchmark międzyjęzykowy wymaga kontrolowanego środowiska, dlatego kompilację uruchamiamy świadomie.


## **20. C z optymalizacją `-O3`**

Typowe polecenie kompilacji dla eksperymentu może wyglądać podobnie do:

```sh
gcc -O3 -shared -fPIC sum.c -o libsum.so
```

Na macOS lub Windows forma biblioteki i polecenia może być inna.


## **21. `-ffast-math` — ważne zastrzeżenie**

Opcja:

```text
-ffast-math
```

pozwala компиляторowi C stosować agresywne optymalizacje arytmetyki zmiennoprzecinkowej.

Może to zmienić zachowanie względem ścisłych reguł IEEE 754.


### Важно

Porównanie:

- C `-O3`,
- C `-O3 -ffast-math`

nie jest porównaniem identycznych gwarancji numerycznych.

Wynik времяowy musi być interpretowany razem z różnicą semantyki.


## **22. Python built-in `sum`**

Oryginalny materiał wykorzystywał połączenie Julia ↔ Python i porównywał wbudowany `sum`.

W nowej wersji zachowujemy ten eksperyment jako część opcjonalną, ponieważ wymaga skonfigurowanego środowiska Python.


In [ ]:
# Przykład koncepcyjny:
# Python:
#
# data = [...]
# result = sum(data)
#
# Pomiar powinien być wykonany po stronie Python narzędziem timeit
# albo przez poprawnie skonfigurowane środowisko interoperacyjności.


### Важно

Przekazywanie dużej tablicy przez granicę języków może wprowadzić dodatkowy koszt konwersji lub kopiowania.

Benchmark powinien измерять operację, a nie przypadkowy narzut transferu danych.


## **23. NumPy**

NumPy implementuje wiele operacji w skompilowanym кодzie niskiego poziomu.

Dlatego:

```python
numpy.sum(a)
```

jest zasadniczo innym przypadkiem niż ręczna цикл:

```python
for x in a:
    total += x
```


### Dobra metodologia

Porównuj osobno:

1. Python built-in `sum`;
2. NumPy `sum`;
3. ręczną pętlę Python.

To trzy różne modele wykonania.


## **24. Ręczna цикл Python**

Ręczna цикл w CPython wykonuje dużą część pracy przez interpreter, dlatego jej charakterystyka производительностиowa różni się od NumPy oraz Julia.


### Важно

Nie należy na podstawie jednej ręcznej pętli twierdzić, że „Python jest медленный” jako cała platforma.

Python często deleguje obliczenia do bibliotek skompilowanych.


## **25. Jak uczciwie porównywać Julia, C i Python?**

### Lista kontrolna

1. Użyj tych samych danych.
2. Sprawdź poprawność результатu.
3. Oddziel kompilację od wykonania.
4. Nie licz kosztu transferu danych, jeśli nie jest częścią zadania.
5. Używaj odpowiednich narzędzi benchmarkowych.
6. Powtarzaj pomiary.
7. Raportuj środowisko.
8. Raportuj wersje компиляторa i bibliotek.
9. Wyjaśnij różnice semantyczne, np. `-ffast-math`.
10. Nie wybieraj tylko результатu korzystnego dla oczekiwanej tezy.


## **26. Column-major order i cache locality**

Julia przechowuje матрицаe kolumnami.

Dla:


In [ ]:
A = rand(1000, 1000)

function sum_by_columns(A)
    total = zero(eltype(A))
    for j in axes(A, 2)
        for i in axes(A, 1)
            total += A[i, j]
        end
    end
    return total
end

function sum_by_rows(A)
    total = zero(eltype(A))
    for i in axes(A, 1)
        for j in axes(A, 2)
            total += A[i, j]
        end
    end
    return total
end

@assert sum_by_columns(A) ≈ sum_by_rows(A)


### Анализ

Pierwsza wersja przechodzi po pamięci zgodnie z układem column-major i zwykle lepiej wykorzystuje cache CPU.


In [ ]:
# using BenchmarkTools
# @btime sum_by_columns($A)
# @btime sum_by_rows($A)


## **27. Broadcasting i fuzja operacji**

Julia może łączyć wiele operacji broadcastowanych w jedną pętlę.


In [ ]:
x = rand(10^6)
y = rand(10^6)

z = @. 2x + 3y^2 - sin(x)

@show length(z)


Makro `@.` dodaje broadcasting do operacji w wyrażeniu.

Taka fuzja może ograniczyć liczbę tablic tymвремяowych.


## **28. Views i аллокации**

Slicing może создаётć kopię:


In [ ]:
v = rand(10^6)

slice_copy = v[100:200_000]
slice_view = @view v[100:200_000]

@show typeof(slice_copy)
@show typeof(slice_view)


Widok pozwala pracować na fragmencie danych bez kopiowania całej sekcji.


## **29. Benchmark funkcji mutujących**

Jeżeli функция zapisuje результат do istniejącego bufora, benchmark powinien uwzględniać ten model.


In [ ]:
function add_vectors!(dest, x, y)
    @inbounds @simd for i in eachindex(dest, x, y)
        dest[i] = x[i] + y[i]
    end
    return dest
end

xv = rand(10^6)
yv = rand(10^6)
dest = similar(xv)

add_vectors!(dest, xv, yv)
@assert dest ≈ xv .+ yv


In [ ]:
# using BenchmarkTools
# @btime add_vectors!($dest, $xv, $yv)


### Важно

Nie alokuj bufora wewnątrz benchmarku, jeżeli chcesz измерять tylko koszt obliczenia do istniejącego bufora.


## **30. Statystyka результатów benchmarku**

Nie patrzymy tylko na jedną liczbę.

Warto analizować:

- minimum;
- medianę;
- rozrzut;
- liczbę alokacji;
- obecność GC.


### Minimum czy mediana?

- **minimum** bywa przybliżeniem najlepszego времяu przy najmniejszym zakłóceniu;
- **mediana** pokazuje bardziej типowy pomiar.

Obie значения mają sens, ale odpowiadają na inne pytania.


## **31. Pułapki микробенчмаркów**

| Pułapka | Problem |
|---|---|
| jeden pomiar | przypadkowy szum |
| brak warm-up | koszt kompilacji |
| глобальные переменные | dodatkowy narzut / gorsza inference |
| brak interpolacji w `BenchmarkTools` | benchmark nie tego, co zamierzaliśmy |
| porównanie różnych semantyk | nieuczciwy eksperyment |
| optymalizacja usuwająca pracę | sztucznie niski время |
| mierzenie transferu między językami | результат obejmuje coś więcej niż algorytm |
| brak informacji o sprzęcie | słaba reprodukowalność |


## **32. Практика**

### Задание 32.1 — trzy wersje sumy

Zaimplementuj:

- `mysum`,
- `mysum_inbounds`,
- `mysum_simd`.

Sprawdź, czy wszystkie возвращаетją результат przybliżenie równy `sum(a)`.


In [ ]:
# Ваше решение:


### Примерowe rozwiązanie 32.1


In [ ]:
function ex_mysum(values)
    total = zero(eltype(values))
    for x in values
        total += x
    end
    return total
end

function ex_mysum_inbounds(values)
    total = zero(eltype(values))
    @inbounds for i in eachindex(values)
        total += values[i]
    end
    return total
end

function ex_mysum_simd(values)
    total = zero(eltype(values))
    @inbounds @simd for i in eachindex(values)
        total += values[i]
    end
    return total
end

@assert ex_mysum(a) ≈ sum(a)
@assert ex_mysum_inbounds(a) ≈ sum(a)
@assert ex_mysum_simd(a) ≈ sum(a)


### Задание 32.2 — benchmark

Jeżeli masz `BenchmarkTools`, porównaj trzy функции z zadania 32.1.


In [ ]:
# using BenchmarkTools
# @btime ex_mysum($a)
# @btime ex_mysum_inbounds($a)
# @btime ex_mysum_simd($a)


### Задание 32.3 — типe stability

Napisz funkcję, która może zwrócić `Int` albo `String`, a następnie zbadaj ją przez `@code_warnтипe`.


In [ ]:
# Ваше решение:


### Примерowe rozwiązanie 32.3


In [ ]:
function mixed_result(x)
    x >= 0 ? x : "negative"
end

@code_warntype mixed_result(10)


### Задание 32.4 — память

Porównaj:

```julia
v[100:10000]
```

oraz:

```julia
@view v[100:10000]
```

pod kątem alokacji.


In [ ]:
v2 = rand(100_000)

copy_alloc = @allocated v2[100:10_000]
view_alloc = @allocated @view v2[100:10_000]

@show copy_alloc
@show view_alloc


## **33. Мини-проект — raport benchmarkowy**

### Цель

Przygotuj raport porównujący:

1. `sum(a)`;
2. `mysum(a)`;
3. `mysum_simd(a)`;
4. opcjonalnie C;
5. opcjonalnie NumPy;
6. opcjonalnie Python built-in.


### Minimalne dane raportu

| Поле | Пример |
|---|---|
| CPU | model procesora |
| OS | Linux / Windows / macOS |
| Julia | wersja |
| rozmiar danych | `10^7 × Float64` |
| narzędzie | BenchmarkTools |
| число próbek | zgodnie z trial |
| результат | minimum / median |
| аллокации | число i rozmiar |


### Важно

Raport powinien zawierać zarówno результат, jak i metodologię.

Bez metodologii число типu „3.2 ms” ma ograniczoną значение naukową.


## **34. Итоговый checkpoint**

Odpowiedz bez uruchamiania кодu:

1. Dlaczego pierwszy przebieg funkcji może być медленнееszy?
2. Czym różnią się `@time` i `@btime`?
3. Po co interpolować `$a` w `BenchmarkTools`?
4. Dlaczego код производительностиowy warto umieszczać w функцияch?
5. Co mierzy `@allocated`?
6. Co oznacza типe stability?
7. Do czego służy `@code_warnтипe`?
8. Co robi `@simd`?
9. Co robi `@inbounds`?
10. Dlaczego dwa результатi `Float64` mogą minimalnie się różnić?
11. Co zmienia `-ffast-math`?
12. Dlaczego NumPy i ręczna цикл Python to różne przypadki?
13. Co oznacza column-major order?
14. Dlaczego kolejność pętli po матрицаy ma znaczenie?
15. Co daje `@view`?
16. Dlaczego minimum i mediana benchmarku odpowiadają na nieco inne pytania?
17. Jakie informacje trzeba raportować razem z результатiem benchmarku?


## **35. Итоги lekcji**

Najważniejsze zasady Lesson 9:

1. Julia może generować bardzo wydajny код maszynowy, ale nie każdy код jest automatycznie быстрый.
2. Pierwsze вызов może zawierać koszt kompilacji JIT.
3. `@time` służy do быстрыйej diagnostyki, a `BenchmarkTools` do dokładniejszych pomiarów.
4. W benchmarkach `BenchmarkTools` należy świadomie interpolować dane przez `$`.
5. Kod krytyczny производительностиowo powinien zwykle znajdować się w функцияch.
6. Alokacje i стабильность типов mają duże znaczenie.
7. `@code_warnтипe` pomaga diagnozować problemy inference.
8. Ręczne циклы Julia mogą być bardzo wydajne.
9. `@simd` i `@inbounds` są narzędziami zaawansowanymi i wymagają poprawnej semantyki.
10. `Float64` ma ograniczoną precyzję, a kolejność sumowania może zmieniać ostatnie bity результатu.
11. C z `-ffast-math` może mieć inne gwarancje numeryczne niż код bez tej flagi.
12. Porównania Julia/C/Python/NumPy muszą uwzględniać rzeczywisty model wykonania.
13. Julia używa układu column-major, dlatego kolejność dostępu do матрицаy wpływa na cache locality.
14. Broadcasting może ograniczać liczbę obiektów tymвремяowych dzięki fuzji.
15. Profesjonalny benchmark zawsze opisuje metodologię, środowisko i ograniczenia eksperymentu.


## Источники для дальнейшего изучения

- Julia Manual — Performance Tips
- Julia Manual — Functions
- Julia Manual — Types
- Julia Manual — Arrays
- Julia Manual — C Interface
- Julia Base — `@time`, `@allocated`
- Julia InteractiveUtils — `@code_warnтипe`
- BenchmarkTools.jl documentation


---

**Cartesian School · Julia Course**  
**Lesson 9 — Julia is Fast**  
**Автор:** Siergej Sobolewski  
**Copyright:** © 2026 Cartesian School

[← Lesson 8 — Multiple Dispatch](Lesson_8_Multiple_Dispatch_Julia_Cartesian_School_RU.ipynb)  
[Оглавление](../README.ru.md)  
[Lesson 10 — Linear Algebra Concepts →](Lesson_10_Linear_Algebra_Concepts_Julia_Cartesian_School_RU.ipynb)
